In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


In [ ]:
# Import TopologicPy modules
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.Dictionary import Dictionary


In [ ]:
selectors = []
ground = CellComplex.Prism(width=20, length=20, height=1, placement="lowerleft", uSides=6, vSides=6, wSides=1)
ground = Topology.Translate(_as_topology(ground), 0, 0, -1)
ground_cells = Topology.Cells(ground)
d = Dictionary.ByKeyValue("id", 0)
for cell in ground_cells:
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)
Topology.Show(ground, xAxis=True, yAxis=True, zAxis=True)


In [ ]:
columns = []
for i in range(2,20,4):
    for j in range(2,20,4):
        origin = Vertex.ByCoordinates(_as_topology(i), _as_topology(j), 0)
        column = Cell.Prism(width=0.8, length=0.8, height=3, origin=origin, placement="bottom")
        columns.append(column)
columns = Cluster.ByTopologies(_as_topology(columns))

column_cells = Topology.Cells(columns)
d = Dictionary.ByKeyValue("id", 1)
for cell in column_cells:
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

Topology.Show(columns, ground)

In [ ]:
origin = Vertex.ByCoordinates(0.5,0.5,3)
podium = CellComplex.Prism(width=19, length=19, height=3, uSides=4, vSides=4, wSides=1, origin=origin, placement="lowerleft")

podium_cells = Topology.Cells(podium)
d = Dictionary.ByKeyValue("id", 2)
for cell in podium_cells:
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

Topology.Show(ground, columns, podium)

In [ ]:
origin = Vertex.ByCoordinates(1,1,6)
block = CellComplex.Prism(width=18, length=18, height=15, uSides=6, vSides=6, wSides=5, origin=origin, placement="lowerleft")
block_cells = Topology.Cells(block)
d = Dictionary.ByKeyValue("id", 3)
for cell in block_cells:
    s = Topology.InternalVertex(cell)
    s = Topology.SetDictionary(s, d)
    selectors.append(s)

Topology.Show(ground, columns, podium, block)

In [ ]:
origin = Vertex.ByCoordinates(7,7,0)
core = Cell.Prism(width=6, length=6, height=24, origin=origin, placement="lowerleft")
s = Topology.InternalVertex(core)
d = Dictionary.ByKeyValue("id", 4)
s = Topology.SetDictionary(s, d)
selectors.append(s)
Topology.Show(ground, columns, podium, block, core)

In [ ]:
columns = Topology.Difference(columns, core)
podium = Topology.Difference(podium, core)
block = Topology.Difference(block, core)


In [ ]:
ground_cells = Topology.Cells(ground)
column_cells = Topology.Cells(columns)
podium_cells = Topology.Cells(podium)
block_cells = Topology.Cells(block)

all_cells = ground_cells+column_cells+podium_cells+block_cells+[core]
building = CellComplex.ByCells(all_cells)

In [ ]:
Topology.Show(building, selectors, faceOpacity=0.1)

In [ ]:
building = Topology.TransferDictionariesBySelectors(building, selectors, tranCells=True)

In [ ]:
graph = TGraph.ByTopology(building)

TGraph.Show(graph, vertexSize=12, vertexGroupKey="id")

In [ ]:
status = TGraph.ExportToCSV(graph, defaultGraphLabel=5, path='C:\\Users\\sarwj\\OneDrive - Cardiff University\\eCAADe 2024 Workshop\\datasets\\test01', nodeLabelKey='id', overwrite=True)
print(status)